In [19]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
device=torch.device("cpu")
print(device)

cpu


In [20]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)
model.eval()


Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4557.93it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [21]:
print(type(model))

<class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


In [22]:
for name, obj in model.named_modules():
    print(name)


shared
encoder
encoder.embed_tokens
encoder.block
encoder.block.0
encoder.block.0.layer
encoder.block.0.layer.0
encoder.block.0.layer.0.SelfAttention
encoder.block.0.layer.0.SelfAttention.q
encoder.block.0.layer.0.SelfAttention.k
encoder.block.0.layer.0.SelfAttention.v
encoder.block.0.layer.0.SelfAttention.o
encoder.block.0.layer.0.SelfAttention.relative_attention_bias
encoder.block.0.layer.0.layer_norm
encoder.block.0.layer.0.dropout
encoder.block.0.layer.1
encoder.block.0.layer.1.DenseReluDense
encoder.block.0.layer.1.DenseReluDense.wi_0
encoder.block.0.layer.1.DenseReluDense.wi_1
encoder.block.0.layer.1.DenseReluDense.wo
encoder.block.0.layer.1.DenseReluDense.dropout
encoder.block.0.layer.1.DenseReluDense.act
encoder.block.0.layer.1.layer_norm
encoder.block.0.layer.1.dropout
encoder.block.1
encoder.block.1.layer
encoder.block.1.layer.0
encoder.block.1.layer.0.SelfAttention
encoder.block.1.layer.0.SelfAttention.q
encoder.block.1.layer.0.SelfAttention.k
encoder.block.1.layer.0.SelfAt

In [23]:
print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [24]:
for p in model.parameters():
    print(p.shape)

torch.Size([32128, 512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([512, 384])
torch.Size([32, 6])
torch.Size([512])
torch.Size([1024, 512])
torch.Size([1024, 512])
torch.Size([512, 1024])
torch.Size([512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([512, 384])
torch.Size([512])
torch.Size([1024, 512])
torch.Size([1024, 512])
torch.Size([512, 1024])
torch.Size([512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([512, 384])
torch.Size([512])
torch.Size([1024, 512])
torch.Size([1024, 512])
torch.Size([512, 1024])
torch.Size([512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([512, 384])
torch.Size([512])
torch.Size([1024, 512])
torch.Size([1024, 512])
torch.Size([512, 1024])
torch.Size([512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([384, 512])
torch.Size([512, 384])
torch.Size([512])
torch.Size([1024, 512])
torch.Size([1024, 512

In [25]:
# 'lm_head' weight is tied to 'shared' weight that's why we see only one entry as torch.Size([32128, 512]) and not two.

In [26]:
input_text = '''summarize: If you wished to know a system, would you study its chaos, or consider an averaged version? 
The answer often feels contradicting to me. Physicists (especially in Statistical Mechanics) would ignore the chaos. But in their description, my life would be simply: “…, cleared JEE advanced, BTech in AI at IIT Kharagpur with 9.8 cgpa, …”. You are missing out on his late night exam stress or the day he didn’t buy chicken because not securing an internship made it feel costlier. Is it fair to remember life just through its curated checkpoints?

A couple of days back, at General Relativity class, a batchmate asked me if I had studied it beforehand. I told him the smoothened answer: “yes, in the summer”. While answering him, a flashback of memory hit my mind. I knew what I was hiding in that “yes”. Coping from breakup, feeling of self doubt because of not securing any internship, conflict with cousins, ranting on Reddit and making my parents tensed too: you see, all of these can be “cancelled” by the simple “yes” reply, but is it that simple? Do we only want to see the “polished” version?

Human memory is optimised in the way that we don't store all the events, rather keep only the major ones and reconstruct the remaining. This reconstruction often leads to breaking of the causal order, i.e., memories become a jigsaw puzzle, not necessarily ordered monotonically by time. This is beautifully shown in the movie “We live in time (2024)” which through its non-linear narrative tells us that our perception of a life lived looks scattered, connecting past joy directly with future grief.

In spite of this “causal reconstruction error”, this description is still better than the “cleared JEE advanced” smoothened version. We are leaving out some chaos, but retaining at least the essence of the story. However, if you are someone’s close one: mom, dad, friend or partner, you shouldn’t be tired listening to the chaos, should you?

My description of life now will contain more chaos from my current timeline rather than the one 4 months back, because the latter is reconstructed. I can't and don't want to replicate my state of mind back then before writing those Reddit posts, and maybe the future me can't copy the current context too. And it's okay, given that we accept our critical memory optimisation.

So what am I currently experiencing? Attending classes 28 hours a week, with classes regularly starting at 8AM. In 3 out of 5 days, I have to run back and forth from one building to another, often leaving class 5min early just to arrive on time for the next, because of the different subjects I took. Along with difficult theory subjects like Statistical Learning Theory, Reinforcement Learning, LLMs, I am going through a very difficult lab this semester: Deep learning for sequential data, preparing for which consumes the entire weekend. I have assignments and homeworks to complete and a self study to do, which I am forced to postpone. I often ask myself: half of my stress is because of my peers being too good; Had they been mediocre, things would have been a lot simpler for me. But what’s then the point of me being at IIT? Competition is its biggest asset.

Amidst this stress, we had a Foundation Day celebration on 29th August. It provided a crucial reboot to my system. Plaban Sir and everyone involved were simply excellent in managing the entire event so effortlessly. And honestly, I feel a little sad that we didn't step forward to take that responsibility ourselves. Taking responsibility is not merely a duty, it is a skill that shapes who we become and hopefully we will try to take it up next year.
'''
inputs = tokenizer(input_text, return_tensors="pt")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (867 > 512). Running this sequence through the model will result in indexing errors


In [27]:
prompt = "summarize: " + input_text.strip()
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512,
)

summary_ids = model.generate(
    inputs["input_ids"].to(device),
    attention_mask=inputs["attention_mask"].to(device),
    max_new_tokens=120,
    min_length=30,
    num_beams=4,
    length_penalty=1.5,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

output_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Summary:", output_text)


Summary: Physicists would ignore the chaos. But in their description, my life would be simply: “..., cleared JEE advanced, BTech in AI at IIT Kharagpur with 9.8 cgpa, ...”


In [28]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name='google/flan-t5-small'
tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4464.78it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [29]:
prompt = "summarize: " + input_text.strip()
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512,
)

summary_ids = model.generate(
    inputs["input_ids"].to(device),
    attention_mask=inputs["attention_mask"].to(device),
    max_new_tokens=120,
    min_length=30,
    num_beams=4,
    length_penalty=1.5,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

output_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Summary:", output_text)


Summary: Physicists would ignore the chaos. But in their description, my life would be simply: “..., cleared JEE advanced, BTech in AI at IIT Kharagpur with 9.8 cgpa, ...”
